# tSCS EMG — ARC-EX (10 kHz carrier, 30 Hz), before vs with lidocaine (P04, 17-07-2026)

## What an ARC-EX ("Modulated") file is
ARC-EX-type stimulation: a **10 kHz carrier** switched on in packets at a **30 Hz envelope** —
each packet is 10 pulses of 50 µs (≈ 1 ms), packets 33.4 ms apart, train = 1000 ms = **30
packets**. Every packet evokes its own response, so the train is analysed exactly like the
30 Hz burst: "pulse *k*" below means "packet *k*". One file = one recruitment sweep = one train
per intensity (5 mA steps, 10 → 110 mA). So a *train* and an *intensity* are the same thing.

## What is measured — fully automatic, nothing picked by hand
For each train, the first **`N_PULSES`** pulses are analysed. For pulse *k*:

    window_k = [ pulse_k onset + RESP_START_MS ,  pulse_(k+1) onset − GUARD_MS ]
    p2p_k    = max(EMG) − min(EMG) inside window_k          (mV)

`RESP_START_MS` exists because the stimulus artifact outlasts the trigger pulse — by a different
amount on every channel. The diagnostics cell prints the measured artifact width per channel.

## Motor-response criterion (`MIN_SNR`)
Automatic, per train (= per muscle × intensity). Two numbers are compared:

1. **signal** — the pulse-1 peak-to-peak, measured in `window_1` (8 → 32.4 ms after the pulse)
2. **noise** — the peak-to-peak of the *pre-stimulus baseline* of that same trace, measured over
   windows of the **same length** (24.4 ms) placed between −95 and −5 ms, averaged

A train **has a motor response if signal ≥ `MIN_SNR` × noise** (default 3). In words: *the first
pulse must evoke something at least three times bigger than what the same window shows when
nothing is stimulated.* Trains that fail are marked *below criterion* and excluded from every
number. Pulse 1 is used because it is the only pulse not affected by depression from a previous
pulse. Only pulse 1 is tested — pulses 2–N of a passing train are kept whatever their size.

## Artifact rejection (`MAX_EDGE_FRAC`)
Some channels (Deltoid, Biceps, Triceps) show no EMG wave between pulses, only the smooth
recovery of the stimulus artifact. Such a curve has no peak inside the window: its max and min
fall **on the window borders**. So a train is **rejected as artifact when more than
`MAX_EDGE_FRAC` (50 %) of its pulses have their max or min within 1 ms of a border.** It is then
treated like a non-responding train and labelled *ARTIFACT – rejected* in the figures.

## Design of this notebook
**Nothing is averaged across intensities.** Each intensity is compared on its own,
before vs with lidocaine, with the raw traces and the detected peaks visible in every figure.

Files: lidocaine was given at **15:14** (`P04_2026-07-17_lidocaine.xlsx`). Electrode 1: `150657` (pre,
15:06) vs `162347` (post, 16:23) · Electrode 3: `151033` (pre) vs `162548` (post; the log says
"electrode 2" for this one, the file says 3). Motor threshold from the log: **100 mA** before /
**105 mA** with lidocaine (elec 1), **140 mA** (elec 3). Intensities are 2–3× the burst ones —
10 kHz packets need far more current to recruit.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from functions import set_style, load_run, detect_pulses, pretty, waterfall, waterfall_overlay, thresholds_for, typical_threshold, threshold_source
from functions.burst import (diagnostics, compare_at_intensity, summary_curves,
                             burst_p2p, save_burst_csv, motor_threshold,
                             plot_pulse_overlay, detection_report)
set_style()


## 1 · Config

In [ ]:
D = "tSCS_CHUV_data/17-07-2026/P04tscsHealthy/"
ELECTRODE = 1          # <-- 1 or 3: picks the file pair and the motor threshold below

FILES = {   # electrode: (before lidocaine, with lidocaine, motor threshold mA from the log)
    1: ("Modulated_autosave_20260717_150657_684ms.csv",   # 15:06
        "Modulated_autosave_20260717_162347_270ms.csv",   # 16:23
        100),                                             # 100 before / 105 with lidocaine
    3: ("Modulated_autosave_20260717_151033_569ms.csv",   # 15:10
        "Modulated_autosave_20260717_162548_293ms.csv",   # 16:25  (log says "electrode 2", file says 3)
        140),                                             # above the sweep -> no MT figure
}
CSV_BEFORE, CSV_AFTER = (D + f for f in FILES[ELECTRODE][:2])

N_PULSES      = 10     # first N pulses of each train
RESP_START_MS = 8.0    # response window starts this long after EACH pulse onset (must clear the artifact)
                       # can also be a dict to override single muscles after the reliability check, e.g.
                       # RESP_START_MS = {"R_DELmed": 11.0}  -> 11 ms for that channel, 8 ms for the rest
                       # (channel names = the keys of `sig`; the default for the rest is 8.0)
GUARD_MS      = 1.0    # ...and stops this long before the next pulse
SNR_ON        = "median" # which pulses the criterion looks at: "median" (the train's median
                        # pulse), "p1" (pulse 1 only), "half" (at least half the pulses).
                        # "p1" throws away a whole train when its first pulse happens to be small.
MIN_SNR       = 1.2    # motor-response criterion: pulse-1 p2p >= MIN_SNR x baseline p2p (None = keep all)
ANCHOR, ANCHOR_WIN_MS = "mean", 3.0   # anchor each pulse's max/min to the train average
MAX_EDGE_FRAC = 0.5    # ARTIFACT rejection: a train is thrown out when more than this fraction of its
                       # pulses have their max/min sitting on the window border (= smooth artifact
                       # recovery, no EMG wave - Deltoid / Biceps / Triceps here). None = off
EDGE_MS       = 1.0    # "on a window border" means within this many ms of it
JITTER_MS     = 0.5    # flag a pulse whose peak latency differs from the train
                       # median by more than this (floored at the anchor window)

KW = dict(n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, guard_ms=GUARD_MS, min_snr=MIN_SNR,
          max_edge_frac=MAX_EDGE_FRAC, snr_on=SNR_ON,
          anchor=ANCHOR, anchor_win_ms=ANCHOR_WIN_MS)

meta_b, t_b, sig_b = load_run(CSV_BEFORE)
meta_a, t_a, sig_a = load_run(CSV_AFTER)
muscles = [c for c in sig_b if c != "Trigger A" and c in sig_a]
amps_b  = [m["amp_ma"] for m in meta_b]
amps_a  = [m["amp_ma"] for m in meta_a]
AMPS    = [a for a in amps_b if a in amps_a]      # intensities present in BOTH files

print("BEFORE:", CSV_BEFORE.split("/")[-1], "| intensities", amps_b)
print("AFTER :", CSV_AFTER.split("/")[-1],  "| intensities", amps_a)
# motor thresholds: picked by hand in notebooks/motor_thresholds/, detected where not picked yet
MT     = [thresholds_for(f, **KW) for f in (CSV_BEFORE, CSV_AFTER)]   # {muscle: mA} each
AMP_MT = typical_threshold(CSV_BEFORE, **KW) or max(AMPS)   # ONE intensity for the diagnostics
                                                        # (was hand-typed: AMP_MT        = FILES[ELECTRODE][2]   # motor threshold (mA) from the log; highlighted if inside the sweep)
AMP_SHOW = AMP_MT if AMP_MT in AMPS else max(AMPS)   # intensity used for the diagnostic / overlay figures
print(f"ELECTRODE {ELECTRODE} | compared at: {AMPS} mA | motor threshold: {AMP_MT} mA"
      + ("" if AMP_SHOW == AMP_MT else f"  (MT outside the sweep -> diagnostics shown at {AMP_SHOW} mA)"))


## 2 · Diagnostics — is the detection sound?

Print-only, one block per file. Read it top to bottom:

1. **artifact width per channel** — `RESP_START_MS` must be larger than these, otherwise the
   "response" is artifact decay. A `!!` line tells you which channel it does *not* clear.
2. **trains kept** per muscle — how many intensities pass the motor-response criterion and from
   which mA. This is the data-derived motor threshold *per muscle*.
3. **CLIPPED** — a channel whose amplifier saturated. Its peak-to-peak is meaningless.

Known for this session: the proximal channels (Deltoid, Biceps, Triceps) show a smooth
artifact-recovery curve between pulses rather than a discrete response — they pass the criterion
on artifact alone and should not be interpreted. **Thenar (R)** clips after lidocaine.

In [ ]:
print("=" * 30, "BEFORE lidocaine", "=" * 30)
res_b = diagnostics(meta_b, t_b, sig_b, muscles, amp=AMP_SHOW, **KW)
print()
print("=" * 30, "WITH lidocaine", "=" * 32)
res_a = diagnostics(meta_a, t_a, sig_a, muscles, amp=AMP_SHOW, **KW)

print("\nsaved", save_burst_csv(res_b, muscles, CSV_BEFORE, meta=meta_b, normalize="none"))
print("saved", save_burst_csv(res_a, muscles, CSV_AFTER,  meta=meta_a, normalize="none"))


## 3 · Reliability of the automatic peak-to-peak

Picking every peak by hand is not feasible (10 pulses × 15 muscles × 9 intensities × 2 files),
and an unchecked automatic pick is the risk. So the detection stays automatic and is **checked in
two ways**, then **corrected per muscle**, not per peak.

### 3a · Pulse overlay — the visual check
For one intensity, every pulse's segment is cut out and **re-aligned to its own pulse onset**
(t = 0), then all N are overlaid; colour = pulse number (dark = 1, light = N). Green = the response
window, ▼/▲ = the detected max/min of each pulse.

**If the detection is picking the same deflection every time, the markers stack on top of each
other.** A marker off on its own = a misdetection, and it gets a **red ring**. The panel title
turns red with the count.

### 3b · Automatic flags — the numeric check
Every detected max/min is tested on all responding trains:

- **edge** — it sits within 1 ms of a window border. The "peak" is the artifact tail, the next
  pulse's onset, or a wave the window cuts off. *A muscle where almost every pulse is "edge" is
  not giving you an EMG response — it is artifact recovery* (the proximal channels here).
- **jitter** — its time after its own pulse differs by > 3 ms from the train's median: it is not
  the same deflection as the other pulses (noise spike, movement, a different wave).

A muscle is trustworthy when its flagged count is low and the overlay stacks. Where a real
response is visible but the window cuts it, override `RESP_START_MS` for that channel in the
config (dict form) and re-run — that is the semi-manual step.

In [ ]:
plot_pulse_overlay(meta_b, t_b, sig_b, muscles, amp=AMP_SHOW, title=f"{AMP_SHOW} mA - before lidocaine", edge_ms=EDGE_MS, jitter_ms=JITTER_MS, **KW)
plot_pulse_overlay(meta_a, t_a, sig_a, muscles, amp=AMP_SHOW, title=f"{AMP_SHOW} mA - with lidocaine", edge_ms=EDGE_MS, jitter_ms=JITTER_MS, **KW);


In [ ]:
print("=" * 30, "BEFORE lidocaine", "=" * 30)
detection_report(res_b, muscles, edge_ms=EDGE_MS, jitter_ms=JITTER_MS)
print()
print("=" * 30, "WITH lidocaine", "=" * 32)
detection_report(res_a, muscles, edge_ms=EDGE_MS, jitter_ms=JITTER_MS);


## 4 · Raw traces — waterfalls, −20 to 80 ms

One trace per intensity stacked at its amplitude, first **3 pulses** of the train (0, 33.4,
66.7 ms), red = artifact of each pulse. **Same gain per muscle in both figures** — the first call
returns the gains and the second reuses them, so a smaller response after lidocaine *draws*
smaller.

### 4a · Before lidocaine

In [ ]:
gains = waterfall(meta_b, t_b, sig_b, muscles, xlim=(-20, 130))


### 4b · With lidocaine — same gain as above

In [ ]:
waterfall(meta_a, t_a, sig_a, muscles, xlim=(-20, 130), gains=gains);


### 4c · Overlay — before (gray) and with lidocaine (orange) on the same panels, same gain

In [ ]:
waterfall_overlay(meta_b, t_b, sig_b, meta_a, t_a, sig_a, muscles, xlim=(-20, 130), gains=gains);


## 5 · Every intensity, one figure each — traces + detected peaks + per-pulse peak-to-peak

Per muscle, for the intensity in the figure title:

- **top** — the two EMG traces overlaid, **gray = before**, **orange = with lidocaine**.
  Red band = artifact of each pulse, green band = the response window.
  **▼ = the max, ▲ = the min** actually used for each pulse's peak-to-peak, in the trace's colour —
  this is the check that the right peaks are being detected.
- **middle** — peak-to-peak of *exactly those two traces*, pulse by pulse, **normalised to the
  before-lidocaine pulse 1 = 100 %** (dotted line). The same reference is used for both
  conditions, so the gray pulse-1 bar is 100 % by construction and every other bar — later gray
  pulses *and* all orange pulses — reads as a fraction of it. Dashed line = mean over the N pulses.
  **No error bars: one train per condition, nothing is averaged.** (`NORM_BARS = "none"` → mV.)
- **bottom** — the same numbers reduced to two bars: **pulse 1** vs the **mean of pulses 2–N**,
  same 100 % reference. The error bar on the mean is **± SD over pulses 2–N of that train**;
  pulse 1 is a single value so it has none. Orange pulse 1 below 100 % = lidocaine reduced the
  first response; mean below pulse 1 = the train depresses.
- *below criterion* = no motor response; *ARTIFACT – rejected* = only artifact recovery in the
  window (see intro); *CLIPPED* = saturated channel.

The motor-threshold intensity (`AMP_MT`) is marked in its title. Figures are in increasing mA.

In [ ]:
NORM_BARS = "before_first"   # bars as % of the BEFORE-lidocaine pulse 1 (same reference for both)
                             # "none" -> mV instead

for a in AMPS:
    tag = f"{a} mA" + ("   ← motor threshold" if a == AMP_MT else "")
    compare_at_intensity(CSV_BEFORE, CSV_AFTER, amp=a, normalize=NORM_BARS, title=tag, **KW)


## 5b · Focus — one muscle, one intensity

Pick a muscle (by its label, e.g. `"Flex. carpi rad. (R)"`) and an intensity, and get the whole
analysis for that single case at full size:

1. the two traces (gray = before, orange = with lidocaine) with the response windows and the
   detected ▼/▲, per-pulse bars, pulse 1 vs mean 2–N — same layout as §5, one wide panel
2. the pulse overlay for each condition — the reliability check for exactly this case

`XLIM_FOCUS` sets the time range of the trace panel (default = the whole analysed train).

In [ ]:
MUSCLE     = "Ext. digitorum (R)"   # any label from the panel titles, or a channel name like "R_The_ED_FD_FCR D"
AMP_FOCUS  = AMP_SHOW                 # mA (must exist in both files)
XLIM_FOCUS = None                     # e.g. (-20, 120) for the first 4 pulses; None = whole analysed train

compare_at_intensity(CSV_BEFORE, CSV_AFTER, amp=AMP_FOCUS, normalize=NORM_BARS, muscles=MUSCLE,
                     xlim=XLIM_FOCUS, title=f"{MUSCLE} - {AMP_FOCUS} mA", **KW)
plot_pulse_overlay(meta_b, t_b, sig_b, MUSCLE, amp=AMP_FOCUS, title=f"{AMP_FOCUS} mA - before lidocaine", edge_ms=EDGE_MS, jitter_ms=JITTER_MS, **KW)
plot_pulse_overlay(meta_a, t_a, sig_a, MUSCLE, amp=AMP_FOCUS, title=f"{AMP_FOCUS} mA - with lidocaine", edge_ms=EDGE_MS, jitter_ms=JITTER_MS, **KW);


## 6 · Summary of all intensities — recruitment curves, nothing averaged

Per muscle, **x = stimulation intensity**. Gray = before lidocaine, orange = with lidocaine.

**Top row — recruitment.**
- **solid line, ● = pulse 1** peak-to-peak in mV. This is the classic recruitment curve.
- **dashed line, ▲ = mean of pulses 2–10** of the same train.
- The **vertical gap between ● and ▲ is the depression along the train**, in mV.
- **hollow ○ = below criterion**: no motor response at that intensity (drawn so you see where
  the threshold is; not used anywhere else).

**Bottom row — depression ratio.** Mean of pulses 2–10 as a % of pulse 1, only for trains with a
response. Dotted line = **100 % = no change**; below it the train depresses, above it facilitates.
If gray and orange overlap, lidocaine did not change how the train behaves.

How to read one muscle: Flex. digitorum (R) — ● rises from 30 mA (threshold) and gray/orange
overlap (lidocaine did not change recruitment); ▲ sits below ● (depression); the ratio climbs from
~10 % at threshold to ~80 % at 45 mA (less depression at higher intensity), the same in both.

In [ ]:
summary_curves(CSV_BEFORE, CSV_AFTER, **KW);
